# AlexNet

Krizhevsky, Sutskever, Hinton, *ImageNet Classification with Deep Convolutional Neural Networks*, NeurIPS 2012.

Much deeper than LeNet-5, ReLU instead of tanh, dropout in the FC layers. Built for 224x224 ImageNet input; this notebook uses the CIFAR-10-adapted version in `model.py` (same channel progression as the paper, kernel sizes/strides re-tuned for 32x32 input) -- see `papers/README.md`.

This notebook trains an `AlexNet` on real CIFAR-10.

In [ ]:
import sys
sys.path.insert(0, '../..')
sys.path.insert(0, '.')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from cnn_playground.data import load_cifar10
from cnn_playground.device import resolve_device
from cnn_playground.utils.seed import set_seed
from model import AlexNet

set_seed(0)
# device options: 'auto' (default, picks cuda/mps if available), 'cpu', 'cuda', 'mps'
device = resolve_device('auto')
print('device:', device)

In [ ]:
train_ds = load_cifar10(train=True)
test_ds = load_cifar10(train=False)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)
print(len(train_ds), len(test_ds))

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=-1)
            correct += (preds == y).sum().item()
            total += y.shape[0]
    return correct / total

model = AlexNet(num_classes=10).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

history = {'train_loss': [], 'test_acc': []}
epochs = 15
for epoch in range(epochs):
    model.train()
    running = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        opt.step()
        running += loss.item() * x.shape[0]
    history['train_loss'].append(running / len(train_ds))
    history['test_acc'].append(evaluate(model, test_loader))

print(f"final test accuracy: {history['test_acc'][-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history['train_loss']); axes[0].set_title('train loss'); axes[0].set_xlabel('epoch')
axes[1].plot(history['test_acc']); axes[1].set_title('test accuracy'); axes[1].set_xlabel('epoch')
fig.tight_layout()
plt.show()

In [ ]:
classes = ['plane','car','bird','cat','deer','dog','frog','horse','ship','truck']
model.eval()
x, y = next(iter(test_loader))
x, y = x[:8].to(device), y[:8]
with torch.no_grad():
    preds = model(x).argmax(dim=-1).cpu()

mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3,1,1)
std = torch.tensor([0.2470, 0.2435, 0.2616]).view(3,1,1)
fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
for i, ax in enumerate(axes):
    img = (x[i].cpu() * std + mean).clamp(0, 1).permute(1, 2, 0)
    ax.imshow(img)
    ax.set_title(f'{classes[preds[i]]}\n({classes[y[i]]})', fontsize=9)
    ax.axis('off')
fig.tight_layout()
plt.show()